# 産業連関分析：輸入石油価格の波及効果と石油投入係数の変動

**原典**: RATS (Regression Analysis of Time Series) プログラム → Python 変換

## 分析内容
| 章 | 内容 |
|---|---|
| I | 輸入石油価格の国内価格への波及効果 (Spread Effect) |
| II-1 | 石油消費の変動・要因分解 |
| II-2 | 石油投入係数の変動（代替効果・省エネ効果） |

---
### 使い方
1. セルを上から順に実行してください（`Shift + Enter`）
2. 実データを使う場合は **「データ読み込み設定」セル** でパスを変更してください
3. グラフは自動で表示されます

## 0. セットアップ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print('NumPy version:', np.__version__)
print('Matplotlib version:', matplotlib.__version__)
print('セットアップ完了')

## 1. データ読み込み設定

実データを使う場合は `USE_REAL_DATA = True` にして、各ファイルパスを設定してください。

In [ ]:
# ============================================================
# データ設定（ここを変更してください）
# ============================================================
USE_REAL_DATA = False  # True にすると実データを読み込む

DATA_DIR = './data/'   # データファイルのディレクトリ

FILES = {
    'gulf'    : DATA_DIR + 'GULF.DAT',     # 28×28 投入係数行列等
    'mto60'   : DATA_DIR + 'MTO60.DAT',    # 46×44 産業連関表（基準年）
    'mto65'   : DATA_DIR + 'MTO65.DAT',    # 46×44 産業連関表（比較年）
    'rio69n70': DATA_DIR + 'RIO69.N70',    # 79×84 産業連関表（1970年）
    'rio69n80': DATA_DIR + 'RIO69.N80',    # 79×84 産業連関表（1980年）
}

# 産業名リスト（任意、省略するとSec.1, Sec.2... と表示）
SECTOR_NAMES_28 = None  # 例: ['農業','製造業',...]
SECTOR_NAMES_31 = None
SECTOR_NAMES_69 = None

print('設定完了: USE_REAL_DATA =', USE_REAL_DATA)

In [ ]:
# ============================================================
# データ読み込みユーティリティ
# ============================================================

def load_matrix(filepath, shape, fortran=True):
    """テキストファイルを行列として読み込む（Fortran指数表記対応）"""
    values = []
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith(('*','#','!')):
                continue
            for tok in line.split():
                try:
                    values.append(float(tok.replace('D','E').replace('d','e')))
                except ValueError:
                    pass
    total = shape[0] * shape[1]
    if len(values) < total:
        raise ValueError(f'データ不足: {len(values)} 個 < 必要 {total} 個 ({shape})')
    return np.array(values[:total]).reshape(shape)


def make_dummy_data():
    """ダミーデータ生成（実データがない場合）"""
    np.random.seed(42)
    data = {}
    # I. 波及効果用
    A = np.random.rand(28,28) * 0.1
    data['A']   = A
    data['Q']   = np.linalg.inv(np.eye(28) - A)
    data['E']   = np.random.rand(28,1)
    data['F']   = np.random.rand(28,1) * 2
    data['M']   = np.random.rand(28,1) * 0.1
    data['GDO'] = np.random.rand(28,1) * 100
    # II-1. 石油消費用
    data['XIJO'] = np.random.rand(46,44) * 10
    data['XIJ1'] = np.random.rand(46,44) * 12
    # II-2. 投入係数変動用
    data['Xij0'] = np.random.rand(79,84) * 10
    data['Xij1'] = np.random.rand(79,84) * 12
    return data


# データ読み込み実行
if USE_REAL_DATA:
    print('実データを読み込みます...')
    raw = {}
    raw['XIJO'] = load_matrix(FILES['mto60'],    (46, 44))
    raw['XIJ1'] = load_matrix(FILES['mto65'],    (46, 44))
    raw['Xij0'] = load_matrix(FILES['rio69n70'], (79, 84))
    raw['Xij1'] = load_matrix(FILES['rio69n80'], (79, 84))
    # GULF.DAT の構造はファイルに応じて調整
    # raw['A'] = load_matrix(FILES['gulf'], (28, 28))
    print('読み込み完了')
else:
    print('ダミーデータを使用します')
    raw = make_dummy_data()

print('データ形状:')
for k, v in raw.items():
    print(f'  {k}: {v.shape}')

---
## I. 輸入石油価格の国内価格への波及効果
### Spread Effect of Import Oil Price on Domestic Prices

$$
\Delta P_{D1} = Q^T \cdot \hat{M} \cdot A^T \cdot \Delta P_M
$$
$$
\Delta P_{D2} = L^T \cdot \hat{M} \cdot A^T \cdot \Delta P_M, \quad
L = [I - (I - \hat{M})A]^{-1}
$$

In [ ]:
# ============================================================
# I. 波及効果の計算
# ============================================================
A   = raw['A']
Q   = raw['Q']
E   = raw['E']
F   = raw['F']
M   = raw['M']
GDO = raw['GDO']

# 国内最終需要・総需要
FD = F - E
XJ = A @ GDO
YI = XJ + FD

# 輸入係数と対角行列
COEFM = M / YI
MHAT  = np.diag(COEFM.flatten())

# レオンチェフ転置・価格変化ベクトル（10%上昇と仮定）
AIJ = A.T
QIJ = Q.T
DPM = np.ones((28,1)) * 0.10

# 直接効果
DDD  = MHAT @ AIJ @ DPM
DPD1 = QIJ @ DDD

# 波及効果（修正レオンチェフ逆行列）
UNITV = np.ones((28,1))
UNITM = np.diag(UNITV.flatten())
LIMAT = np.linalg.inv(UNITM - (UNITM - MHAT) @ A)
DPD2  = LIMAT.T @ DDD

print('=== I. 波及効果 計算完了 ===')
print(f'直接効果 DPD1: 平均={DPD1.mean():.6f}, 合計={DPD1.sum():.6f}')
print(f'波及効果 DPD2: 平均={DPD2.mean():.6f}, 合計={DPD2.sum():.6f}')
print(f'追加効果 (DPD2-DPD1): 合計={( DPD2 - DPD1).sum():.6f}')

In [ ]:
# === グラフ I: 直接効果 vs 波及効果 ===
n = 28
labels = SECTOR_NAMES_28 if SECTOR_NAMES_28 else [f'Sec.{i+1}' for i in range(n)]
x = np.arange(n)
w = 0.4

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('I. Spread Effect of Import Oil Price on Domestic Prices',
             fontsize=13, fontweight='bold')

ax = axes[0]
ax.bar(x-w/2, DPD1.flatten(), w, label='Direct Effect (DPD1)', color='#2196F3', alpha=0.85)
ax.bar(x+w/2, DPD2.flatten(), w, label='Spread Effect (DPD2)', color='#FF5722', alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Price Change'); ax.legend(); ax.grid(axis='y', ls='--', alpha=0.5)
ax.set_title('Price Change by Sector: Direct vs Spread Effect')

ax2 = axes[1]
diff = (DPD2 - DPD1).flatten()
ax2.bar(x, diff, color=['#4CAF50' if v>=0 else '#F44336' for v in diff], alpha=0.85)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Additional Effect'); ax2.grid(axis='y', ls='--', alpha=0.5)
ax2.set_title('Additional Effect via Leontief Inverse (DPD2 - DPD1)')

plt.tight_layout()
plt.show()

---
## II-1. 石油消費の変動・要因分解
### Fluctuation of Petroleum Consumption

$$
\Delta X_p = \underbrace{\Delta\hat{A} \cdot L_0 \cdot F_0}_{\text{投入係数変化}} +
\underbrace{\hat{A}_0 \cdot \Delta L \cdot F_0}_{\text{構造変化}} +
\underbrace{\hat{A}_0 \cdot L_0 \cdot \Delta F}_{\text{最終需要変化}} +
\underbrace{\text{交差項}}_{\text{CROSS}}
$$

In [ ]:
# ============================================================
# II-1a. 基準年の計算
# ============================================================
XIJO = raw['XIJO']

# 投入係数行列
AIJO = np.zeros((31,31))
for j in range(31):
    for i in range(31):
        AIJO[i,j] = XIJO[i,j] / XIJO[j,43]

# 最終需要
CONSO = np.array([[XIJO[i,31]+XIJO[i,32]+XIJO[i,33]+XIJO[i,34]] for i in range(31)])
INVO  = np.array([[XIJO[i,35]+XIJO[i,36]+XIJO[i,37]] for i in range(31)])
EXPO  = np.array([[XIJO[i,38]+XIJO[i,39]+XIJO[i,40]] for i in range(31)])
FDO   = CONSO + INVO + EXPO
FDDO  = CONSO + INVO

# 輸入・国内生産・市場供給
IMPO  = np.array([[-XIJO[i,41]-XIJO[i,42]] for i in range(31)])
XJO   = np.array([[XIJO[i,43]-FDO[i,0]+IMPO[i,0]] for i in range(31)])
DMSTXO = XJO + FDDO

# 輸入係数・対角行列・修正レオンチェフ逆行列
COEFMO = IMPO / DMSTXO
MHATO  = np.diag(COEFMO.flatten())
UNITM  = np.eye(31)
LINVO  = np.linalg.inv(UNITM - (UNITM - MHATO) @ AIJO)

print('基準年計算完了')
print(f'  最終需要 FDO 合計: {FDO.sum():.2f}')
print(f'  輸入係数 COEFMO 平均: {COEFMO.mean():.4f}')

In [ ]:
# ============================================================
# II-1b. 比較年の計算と要因分解
# ============================================================
XIJ1 = raw['XIJ1']

AIJ1  = np.zeros((31,31))
for j in range(31):
    for i in range(31):
        AIJ1[i,j] = XIJ1[i,j] / XIJ1[j,43]

CONS1 = np.array([[XIJ1[i,31]+XIJ1[i,32]+XIJ1[i,33]+XIJ1[i,34]] for i in range(31)])
INV1  = np.array([[XIJ1[i,35]+XIJ1[i,36]+XIJ1[i,37]] for i in range(31)])
EXP1  = np.array([[XIJ1[i,38]+XIJ1[i,39]+XIJ1[i,40]] for i in range(31)])
FD1   = CONS1 + INV1 + EXP1
FDD1  = CONS1 + INV1

IMP1  = np.array([[-XIJ1[i,41]-XIJ1[i,42]] for i in range(31)])
XJ1   = np.array([[XIJ1[i,43]-FD1[i,0]+IMP1[i,0]] for i in range(31)])
DMSTX1 = XJ1 + FDD1

COEFM1 = IMP1 / DMSTX1
MHAT1  = np.diag(COEFM1.flatten())
LINV1  = np.linalg.inv(UNITM - (UNITM - MHAT1) @ AIJ1)

# 要因分解
DAHAT  = AIJ1 - AIJO
DLINV  = LINV1 - LINVO
DELTAF = FD1 - FDO

PETRO = DAHAT @ LINVO @ FDO
STRUC = AIJO  @ DLINV @ FDO
FINAL = AIJO  @ LINVO @ DELTAF
CROSS = (DAHAT @ LINVO @ FDO
       + AIJO  @ DLINV @ DELTAF
       + DAHAT @ LINVO @ DELTAF
       + DAHAT @ DLINV @ DELTAF)

print('=== II-1. 要因分解 計算完了 ===')
print(f'  投入係数変化 PETRO: {PETRO.sum():>12.2f}')
print(f'  構造変化     STRUC: {STRUC.sum():>12.2f}')
print(f'  最終需要変化 FINAL: {FINAL.sum():>12.2f}')
print(f'  交差項       CROSS: {CROSS.sum():>12.2f}')

In [ ]:
# === グラフ II-1: 要因分解 ===
n31 = 31
labels31 = SECTOR_NAMES_31 if SECTOR_NAMES_31 else [f'Ind.{i+1}' for i in range(n31)]
x31 = np.arange(n31)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('II-1. Fluctuation of Petroleum Consumption: Factor Decomposition',
             fontsize=12, fontweight='bold')

# 積み上げ棒グラフ
ax = axes[0]
colors4 = ['#1565C0','#2E7D32','#F57F17','#880E4F']
items4  = [('Input Coeff. (PETRO)', PETRO.flatten()),
           ('Structure   (STRUC)',  STRUC.flatten()),
           ('Final Demand (FINAL)', FINAL.flatten()),
           ('Cross Term  (CROSS)',  CROSS.flatten())]
bp, bn = np.zeros(n31), np.zeros(n31)
for (lbl, vals), col in zip(items4, colors4):
    pv = np.where(vals>0, vals, 0)
    nv = np.where(vals<0, vals, 0)
    ax.bar(x31, pv, bottom=bp, label=lbl, color=col, alpha=0.85)
    ax.bar(x31, nv, bottom=bn, color=col, alpha=0.85)
    bp += pv; bn += nv
ax.axhline(0, color='black', lw=0.8)
ax.set_title('Stacked Decomposition by Industry')
ax.set_ylabel('Change in Petroleum Demand')
ax.set_xticks(x31); ax.set_xticklabels(labels31, rotation=45, ha='right', fontsize=7)
ax.legend(fontsize=8); ax.grid(axis='y', ls='--', alpha=0.4)

# パイチャート
ax2 = axes[1]
totals4 = [abs(PETRO.sum()), abs(STRUC.sum()), abs(FINAL.sum()), abs(CROSS.sum())]
pie_lbl = ['Input Coeff.\n(PETRO)','Structure\n(STRUC)','Final Demand\n(FINAL)','Cross\n(CROSS)']
ax2.pie(totals4, labels=pie_lbl, colors=colors4, autopct='%1.1f%%',
        startangle=90, textprops={'fontsize':9})
ax2.set_title('Relative Share (absolute values)')

plt.tight_layout()
plt.show()

In [ ]:
# === グラフ: レオンチェフ逆行列ヒートマップ ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, mat, title in [
    (axes[0], LINVO, 'Leontief Inverse (Base Year)'),
    (axes[1], LINV1, 'Leontief Inverse (Period-t)'),
]:
    im = ax.imshow(mat, cmap='YlOrRd', aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Industry (column)'); ax.set_ylabel('Industry (row)')
plt.suptitle('Leontief Inverse Matrix Comparison', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## II-2. 石油投入係数の変動分析
### Fluctuation of Petroleum Input Coefficient

$$
\Delta P_{ij} = \underbrace{\Delta R_p \cdot E_{ij0}}_{\text{代替効果 (SUBS)}} +
\underbrace{R_{p0} \cdot \Delta E_{ij}}_{\text{省エネ効果 (REFE)}} +
\underbrace{\Delta R_p \cdot \Delta E_{ij}}_{\text{交差項 (CROS)}}
$$

ここで $P_{ij} = R_p \times E_{ij}$, $R_p$ = 石油/エネルギー比率, $E_{ij}$ = エネルギー強度

In [ ]:
# ============================================================
# II-2. 石油投入係数変動の計算
# ============================================================
Xij0 = raw['Xij0']
Xij1 = raw['Xij1']
n69  = 69

def calc_energy_coeff(Xij, n):
    """エネルギー投入係数を計算"""
    total = Xij[78, :n].reshape(1, n)   # 行79: 総生産額
    Ep    = Xij[33, :n].reshape(1, n)   # 行34: 石油投入
    Et    = (Ep + Xij[34,:n] + Xij[5,:n] + Xij[7,:n]).reshape(1, n)
    Pij   = Ep / total
    Rp    = Ep / Et
    Eij   = Et / total
    Pij_check = Rp * Eij
    return Pij, Rp, Eij, Pij_check

Pij0, Rp0, Eij0, Pij01 = calc_energy_coeff(Xij0, n69)
Pij1, Rp1, Eij1, Pij11 = calc_energy_coeff(Xij1, n69)

# 変化量と要因分解
DPij  = Pij1  - Pij0
DPij1 = Pij11 - Pij01
DRp   = Rp1   - Rp0
DEij  = Eij1  - Eij0

SUBS  = DRp  * Eij0
REFE  = Rp0  * DEij
CROS  = DRp  * DEij
DPij2 = SUBS + REFE + CROS

print('=== II-2. 投入係数変動 計算完了 ===')
print(f'  代替効果 SUBS  平均: {SUBS.mean():>10.6f}')
print(f'  省エネ   REFE  平均: {REFE.mean():>10.6f}')
print(f'  交差項   CROS  平均: {CROS.mean():>10.6f}')
print(f'  合計     DPij2 平均: {DPij2.mean():>10.6f}')
print(f'  検証 (DPij1-DPij2): {(DPij1-DPij2).mean():>10.8f}  ← 0に近いほど正確')

In [ ]:
# === グラフ II-2: 投入係数変動 4パネル ===
labels69 = SECTOR_NAMES_69 if SECTOR_NAMES_69 else [f'Ind.{i+1}' for i in range(n69)]
x69 = np.arange(n69)

fig = plt.figure(figsize=(16, 11))
fig.suptitle('II-2. Fluctuation of Petroleum Input Coefficient', fontsize=13, fontweight='bold')
gs = GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

# パネル1: 散布図 基準年 vs 比較年
ax1 = fig.add_subplot(gs[0,0])
ax1.scatter(Pij0.flatten(), Pij1.flatten(), s=25, alpha=0.7,
            color='#1976D2', edgecolors='white', lw=0.5)
lim = [min(Pij0.min(),Pij1.min())*0.9, max(Pij0.max(),Pij1.max())*1.1]
ax1.plot(lim, lim, 'r--', lw=1, label='No change')
ax1.set_xlabel('Pij (Base Year)'); ax1.set_ylabel('Pij (Period-t)')
ax1.set_title('Input Coefficient: Base vs Period-t'); ax1.legend(fontsize=8)
ax1.grid(ls='--', alpha=0.4)

# パネル2: 要因分解積み上げ
ax2 = fig.add_subplot(gs[0,1])
cols3 = ['#1565C0','#2E7D32','#F57F17']
items3 = [('SUBS (Substitution)', SUBS.flatten()),
          ('REFE (Energy Saving)', REFE.flatten()),
          ('CROS (Cross Term)',    CROS.flatten())]
bp2, bn2 = np.zeros(n69), np.zeros(n69)
for (lbl, v), col in zip(items3, cols3):
    pv = np.where(v>0,v,0); nv = np.where(v<0,v,0)
    ax2.bar(x69, pv, bottom=bp2, color=col, alpha=0.8, label=lbl)
    ax2.bar(x69, nv, bottom=bn2, color=col, alpha=0.8)
    bp2+=pv; bn2+=nv
ax2.plot(x69, DPij.flatten(), 'ko-', ms=2, lw=0.8, label='Actual DPij', zorder=5)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_title('Factor Decomposition of DPij'); ax2.set_ylabel('Change')
ax2.legend(fontsize=7); ax2.grid(axis='y', ls='--', alpha=0.4); ax2.set_xticks([])

# パネル3: 石油比率変化
ax3 = fig.add_subplot(gs[1,0])
drp = DRp.flatten()
ax3.bar(x69, drp, color=['#1565C0' if v>=0 else '#C62828' for v in drp], alpha=0.8)
ax3.axhline(0, color='black', lw=0.8)
ax3.set_title('Change in Oil/Energy Ratio (DRp)'); ax3.set_ylabel('DRp')
ax3.grid(axis='y', ls='--', alpha=0.4); ax3.set_xticks([])

# パネル4: エネルギー強度変化
ax4 = fig.add_subplot(gs[1,1])
deij = DEij.flatten()
ax4.bar(x69, deij, color=['#2E7D32' if v>=0 else '#C62828' for v in deij], alpha=0.8)
ax4.axhline(0, color='black', lw=0.8)
ax4.set_title('Change in Energy Intensity (DEij)'); ax4.set_ylabel('DEij')
ax4.grid(axis='y', ls='--', alpha=0.4); ax4.set_xticks([])

plt.show()

---
## まとめ・結果サマリー

In [ ]:
print('=' * 55)
print('  分析結果サマリー')
print('=' * 55)
print()
print('【I. 輸入石油価格の波及効果】')
print(f'  輸入価格上昇 (仮定): 10%')
print(f'  直接効果 DPD1: 平均 {DPD1.mean()*100:.4f}%')
print(f'  波及効果 DPD2: 平均 {DPD2.mean()*100:.4f}%')
print(f'  乗数効果 (DPD2/DPD1): {(DPD2.sum()/DPD1.sum()):.4f}')
print()
print('【II-1. 石油消費変動の要因分解】')
total_change = PETRO.sum() + STRUC.sum() + FINAL.sum() + CROSS.sum()
for name, val in [('投入係数変化 PETRO', PETRO.sum()),
                   ('構造変化     STRUC', STRUC.sum()),
                   ('最終需要変化 FINAL', FINAL.sum()),
                   ('交差項       CROSS', CROSS.sum())]:
    pct = val / abs(total_change) * 100 if total_change != 0 else 0
    print(f'  {name}: {val:>12.2f}  ({pct:>6.1f}%)')
print()
print('【II-2. 石油投入係数の変動（産業平均）】')
for name, val in [('代替効果 SUBS', SUBS.mean()),
                   ('省エネ効果 REFE', REFE.mean()),
                   ('交差項   CROS', CROS.mean()),
                   ('合計     DPij2', DPij2.mean())]:
    print(f'  {name}: {val:>12.6f}')
print()
print('=' * 55)